# Shotloom on a free Kaggle GPU

Open-source AI video studio (Wan 2.2, Apache-2.0). This notebook runs it on Kaggle's free GPUs.

**One-time setup**
1. Kaggle account → *Settings* → verify your phone number (needed for internet + GPU).
2. In this notebook: right panel → *Session options* → **Accelerator: GPU T4 x2**, **Internet: on**.

**Two ways to use it**
- **BATCH (recommended)** – upload your project as a Kaggle dataset, then *Save Version → Save & Run All (Commit)*.
  It renders every shot in the background for up to 12 h, even with your laptop closed. Download the output folder when it finishes.
- **STUDIO** – opens the full web app with a private link so you can generate shot by shot. Runs only while this tab is open.

**What to upload as a dataset** (*Add Input → Upload*), either:
- a project folder made in Shotloom on your PC (contains `project.json`), **or**
- `shots.csv` + your images (in the same folder or an `images/` subfolder).

First run downloads the model (~34 GB, about 5–10 min on Kaggle).

In [ ]:
# ---------------- settings ----------------
MODE     = "batch"          # "batch" or "studio"
QUALITY  = "standard"       # "draft" (480p, fast), "standard" (704p), "final" (704p, more steps)
ASPECT   = "16:9"           # "16:9" YouTube, "9:16" Shorts/Reels (used when importing a CSV)
STYLE    = ""               # optional text added to every AI prompt
PROJECT  = "my-film"        # project name when importing from a CSV
PASSWORD = "change-me"      # studio mode: login is user 'shotloom' + this password
REPO     = "https://github.com/faisal8143/shotloom.git"

In [ ]:
%%capture install_log
!pip install -q -U "diffusers>=0.35.0" "transformers>=4.50" accelerate ftfy sentencepiece gradio imageio-ffmpeg
!rm -rf /kaggle/working/shotloom && git clone -q --depth 1 $REPO /kaggle/working/shotloom

In [ ]:
import glob, os, subprocess, sys, torch
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "NONE - turn on GPU T4 x2")
os.chdir("/kaggle/working/shotloom")
HOME = "/kaggle/working/projects"
os.makedirs(HOME, exist_ok=True)

def run(args):
    # stream output live so the commit log shows progress
    p = subprocess.Popen([sys.executable, "-m", "shotloom.cli", *args, "--home", HOME],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise SystemExit("shotloom exited with an error - see log above")

# find the input: a Shotloom project folder, or a shots.csv
projects = glob.glob("/kaggle/input/**/project.json", recursive=True)
csvs = [c for c in glob.glob("/kaggle/input/**/*.csv", recursive=True)]
if projects:
    src = os.path.dirname(projects[0]); PROJECT = os.path.basename(src)
    print("using project folder:", src)
    run(["status", "--project", PROJECT, "--copy-from", src])
elif csvs:
    folder = os.path.dirname(csvs[0])
    images = os.path.join(folder, "images") if os.path.isdir(os.path.join(folder, "images")) else folder
    print("importing", csvs[0], "images from", images)
    run(["import", "--project", PROJECT, "--csv", csvs[0], "--images", images,
         "--aspect", ASPECT, "--style", STYLE])
else:
    print("No input found - studio mode will start with an empty project list.")

In [ ]:
if MODE == "batch":
    run(["batch", "--project", PROJECT, "--quality", QUALITY, "--placeholders"])
    run(["build", "--project", PROJECT])
    print("\nDone. Output tab -> projects/" + PROJECT + " (takes/, exports/, project.json)")

In [ ]:
if MODE == "studio":
    # the password goes in through the environment, never the shell, so $ ! " etc. stay intact
    os.environ["SHOTLOOM_HOME"] = HOME
    os.environ["SHOTLOOM_PASSWORD"] = PASSWORD
    p = subprocess.Popen([sys.executable, "-u", "app.py", "--share", "--home", HOME],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:          # streams the public link and logs into this cell
        print(line, end="")


### Getting your clips back
*Output* panel → `projects/<your project>` → download. Drop that folder into the `projects` folder of Shotloom on your PC
to pick takes and build the final film, or use `exports/` directly – the batch already built a rough cut.

Kaggle saves up to 20 GB of output per notebook; a 10-minute film is roughly 1–2 GB of clips.